# 第23章 道具を使わせる：エージェント

LLMに電卓と検索を渡し、自分で考えて使わせる

教科書のページ：https://sakunokyumei.github.io/llm-research-textbook/23-agents.html

上のセルから順に、再生ボタン（▶）か `Shift + Enter` で実行します。説明は教科書のページにあります。

**このノートはGPUを使います。** メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選んでから実行してください。


## 準備：少し大きめのモデル


In [ ]:
!wget -q https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/code/rag_docs.jsonl
import ast, json, operator, re, datetime
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
name = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16 if device == "cuda" else torch.float32).to(device)
print("準備完了")

## 道具を作る


In [ ]:
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
       ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}

def calculator(expression: str) -> str:
    """四則演算の式を正確に計算する。

    Args:
        expression: 計算したい式。例：'123*456' や '(3+4)/2'
    """
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in OPS:
            return OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in OPS:
            return OPS[type(node.op)](ev(node.operand))
        raise ValueError("数字と + - * / ** ( ) 以外は使えません")
    try:
        expression = expression.replace("×", "*").replace("÷", "/")
        return str(ev(ast.parse(expression, mode="eval").body))
    except Exception as e:
        return f"エラー：{e}"

docs = [json.loads(l) for l in open("rag_docs.jsonl", encoding="utf-8")]

def search_school(query: str) -> str:
    """ボンゴ情報専門学校の資料から、質問に関係する文章を探す。

    Args:
        query: 調べたいこと。例：'学食の人気メニュー'
    """
    grams = lambda s: {s[i:i + 2] for i in range(len(s) - 1)}
    q = grams(query)
    best = sorted(docs, key=lambda d: -len(q & grams(d["title"] + d["text"])))[:2]
    return "\n".join(f"[{d['id']}] {d['text']}" for d in best)

def today() -> str:
    """今日の日付を返す。"""
    return datetime.date.today().isoformat()

tools = {"calculator": calculator, "search_school": search_school, "today": today}
print(calculator("123*456"), calculator("(3+4)/2"), calculator("__import__('os')"))

## 道具の説明は、プロンプトに入る


In [ ]:
messages = [{"role": "user", "content": "123×456はいくつ？"}]
prompt = tok.apply_chat_template(messages, tools=list(tools.values()), add_generation_prompt=True, tokenize=False)
print(prompt)

## LLMの「道具を使いたい」を読み取る


In [ ]:
def generate(messages):
    inputs = tok.apply_chat_template(messages, tools=list(tools.values()), add_generation_prompt=True,
                                     return_tensors="pt", return_dict=True).to(device)
    out = llm.generate(**inputs, max_new_tokens=200, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def parse_tool_calls(text):
    calls = []
    # ( ) で囲んだ部分だけを取り出す。re.S は「. が改行にも当てはまる」指定（呼び出しは複数行になるため）
    for block in re.findall(r"<tool_call>(.*?)</tool_call>", text, re.S):
        try:
            calls.append(json.loads(block))
        except json.JSONDecodeError:
            pass                       # 形式が崩れていたら無視する
    return calls

reply = generate(messages)
print("LLMの返事：", reply)
print("読み取った呼び出し：", parse_tool_calls(reply))

## エージェントのループ


In [ ]:
def run_agent(question, max_steps=5, verbose=True):
    messages = [{"role": "user", "content": question}]
    for step in range(max_steps):
        reply = generate(messages)
        calls = parse_tool_calls(reply)
        if not calls:                                       # 道具の呼び出しがない → 答え
            return reply
        messages.append({"role": "assistant", "content": "",
                         "tool_calls": [{"type": "function", "function": c} for c in calls]})
        for c in calls:
            fn = tools.get(c.get("name"))
            try:                                            # 引数がおかしくても止まらないように
                result = fn(**c.get("arguments", {})) if fn else f"そんな道具はありません：{c.get('name')}"
            except Exception as e:
                result = f"エラー：{e}"                     # エラーの内容をLLMに返して、やり直させる
            if verbose:
                print(f"  🔧 {c.get('name')}({c.get('arguments')}) → {result[:60]}")
            messages.append({"role": "tool", "name": c.get("name"), "content": result})
    return "（上限の回数に達しました）"

print(run_agent("123×456はいくつ？"))

In [ ]:
print(run_agent("ボンゴ情報専門学校の学食の一番人気メニューを、3人分買うといくらになりますか？"))